# 03 — In Season Feature Engineering

This notebook creates the weekly performance features used by the in season projection layer.

It does not modify the original preseason notebooks or their outputs.

The purpose of this notebook is to convert completed regular season games into team level offensive and defensive performance features that can later be blended with the frozen preseason model.

For a selected target week, the notebook:

- Loads only completed games from weeks before the target week
- Builds team level scoring and margin summaries
- Separates offense and defense
- Adds simple opponent adjusted performance features
- Applies early season shrinkage to reduce overreaction to small samples
- Saves a clean feature table for the weekly team strength notebook

This notebook is intentionally conservative after Week 1. One game should inform the model, not redefine it.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the outputs created by:

`02_Weekly_Data_Update.ipynb`

The only value that normally changes week to week is `TARGET_WEEK`.


In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 2

completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

print("Using:", completed_games_path)


Using: ..\..\data\processed\weekly\week_02_completed_games.parquet


# Load Completed Games

Only games completed before the target week should exist in this file.

That leakage rule was enforced in `02_Weekly_Data_Update.ipynb`.


In [3]:
completed_games = pd.read_parquet(
    completed_games_path
)

completed_games = completed_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Completed games available before Week {TARGET_WEEK}:",
    len(completed_games)
)

display(
    completed_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)


Completed games available before Week 2: 16


,week,gameday,away_team,away_score,home_team,home_score
0,1,2026-09-09,NE,10.0,SEA,13.0
1,1,2026-09-10,SF,27.0,LA,7.0
2,1,2026-09-13,ARI,26.0,LAC,14.0
3,1,2026-09-13,ATL,13.0,PIT,20.0
4,1,2026-09-13,BAL,41.0,IND,23.0
5,1,2026-09-13,BUF,36.0,HOU,31.0
6,1,2026-09-13,CHI,59.0,CAR,37.0
7,1,2026-09-13,CLE,10.0,JAX,34.0
8,1,2026-09-13,DAL,20.0,NYG,28.0
9,1,2026-09-13,GB,22.0,MIN,39.0


# Convert Games to Team-Game Rows

Each NFL game becomes two rows:

- one from the home team's perspective
- one from the away team's perspective

This makes it much easier to calculate team level offensive and defensive features.


In [4]:
home_rows = pd.DataFrame(
    {
        "game_id": completed_games["game_id"],
        "week": completed_games["week"],
        "team": completed_games["home_team"],
        "opponent": completed_games["away_team"],
        "is_home": 1,
        "points_for": completed_games["home_score"],
        "points_against": completed_games["away_score"]
    }
)

away_rows = pd.DataFrame(
    {
        "game_id": completed_games["game_id"],
        "week": completed_games["week"],
        "team": completed_games["away_team"],
        "opponent": completed_games["home_team"],
        "is_home": 0,
        "points_for": completed_games["away_score"],
        "points_against": completed_games["home_score"]
    }
)

team_games = pd.concat(
    [home_rows, away_rows],
    ignore_index=True
)

team_games["point_margin"] = (
    team_games["points_for"]
    - team_games["points_against"]
)

team_games = team_games.sort_values(
    ["team", "week", "game_id"]
).reset_index(drop=True)

display(team_games.head(10))


,game_id,week,team,opponent,is_home,points_for,points_against,point_margin
0,2026_01_ARI_LAC,1,ARI,LAC,0,26.0,14.0,12.0
1,2026_01_ATL_PIT,1,ATL,PIT,0,13.0,20.0,-7.0
2,2026_01_BAL_IND,1,BAL,IND,0,41.0,23.0,18.0
3,2026_01_BUF_HOU,1,BUF,HOU,0,36.0,31.0,5.0
4,2026_01_CHI_CAR,1,CAR,CHI,1,37.0,59.0,-22.0
5,2026_01_CHI_CAR,1,CHI,CAR,0,59.0,37.0,22.0
6,2026_01_TB_CIN,1,CIN,TB,1,33.0,27.0,6.0
7,2026_01_CLE_JAX,1,CLE,JAX,0,10.0,34.0,-24.0
8,2026_01_DAL_NYG,1,DAL,NYG,0,20.0,28.0,-8.0
9,2026_01_DEN_KC,1,DEN,KC,0,10.0,31.0,-21.0


# League Baselines

The current season league averages provide the reference point for the in season features.

After Week 1, these values are based on only one week, so they are treated cautiously later through shrinkage.


In [5]:
league_points_per_team_game = (
    team_games["points_for"].mean()
)

league_margin_mean = (
    team_games["point_margin"].mean()
)

print(
    "League points per team-game:",
    round(league_points_per_team_game, 3)
)

print(
    "League average margin:",
    round(league_margin_mean, 6)
)


League points per team-game: 24.719
League average margin: 0.0


# Raw Team Performance Summary

The first layer is intentionally simple and interpretable:

- games played
- points scored per game
- points allowed per game
- point differential per game

We also convert offense and defense into "above league average" values.

For offense:

`offense_points_above_avg = PF/G - league scoring average`

Higher is better.

For defense:

`defense_points_saved_vs_avg = league scoring average - PA/G`

Higher is better.

This gives offense and defense the same intuitive direction: positive values are good.


In [6]:
team_summary = (
    team_games
    .groupby("team")
    .agg(
        games_played=("game_id", "count"),
        points_for=("points_for", "sum"),
        points_against=("points_against", "sum"),
        point_margin=("point_margin", "sum")
    )
    .reset_index()
)

team_summary["points_for_per_game"] = (
    team_summary["points_for"]
    / team_summary["games_played"]
)

team_summary["points_against_per_game"] = (
    team_summary["points_against"]
    / team_summary["games_played"]
)

team_summary["point_diff_per_game"] = (
    team_summary["point_margin"]
    / team_summary["games_played"]
)

team_summary["offense_points_above_avg"] = (
    team_summary["points_for_per_game"]
    - league_points_per_team_game
)

team_summary["defense_points_saved_vs_avg"] = (
    league_points_per_team_game
    - team_summary["points_against_per_game"]
)

team_summary["raw_net_points_above_avg"] = (
    team_summary["offense_points_above_avg"]
    + team_summary["defense_points_saved_vs_avg"]
)

display(
    team_summary.sort_values(
        "raw_net_points_above_avg",
        ascending=False
    )
)


,team,games_played,points_for,points_against,point_margin,points_for_per_game,points_against_per_game,point_diff_per_game,offense_points_above_avg,defense_points_saved_vs_avg,raw_net_points_above_avg
14,JAX,1,34.0,10.0,24.0,34.0,10.0,24.0,9.28125,14.71875,24.0
5,CHI,1,59.0,37.0,22.0,59.0,37.0,22.0,34.28125,-12.28125,22.0
15,KC,1,31.0,10.0,21.0,31.0,10.0,21.0,6.28125,14.71875,21.0
28,SF,1,27.0,7.0,20.0,27.0,7.0,20.0,2.28125,17.71875,20.0
2,BAL,1,41.0,23.0,18.0,41.0,23.0,18.0,16.28125,1.71875,18.0
20,MIN,1,39.0,22.0,17.0,39.0,22.0,17.0,14.28125,2.71875,17.0
18,LV,1,27.0,13.0,14.0,27.0,13.0,14.0,2.28125,11.71875,14.0
24,NYJ,1,23.0,10.0,13.0,23.0,10.0,13.0,-1.71875,14.71875,13.0
0,ARI,1,26.0,14.0,12.0,26.0,14.0,12.0,1.28125,10.71875,12.0
23,NYG,1,28.0,20.0,8.0,28.0,20.0,8.0,3.28125,4.71875,8.0


# Opponent Adjustment

Raw Week 1 results can be misleading because each team has faced only one opponent.

A 30 point offensive performance against a strong defense should be viewed differently from the same 30 points against a weak defense.

At this stage, we use a simple first pass opponent adjustment:

- Offensive performance is adjusted by the opponent's defensive result
- Defensive performance is adjusted by the opponent's offensive result

Because early season data is extremely sparse, this opponent adjustment is intentionally modest and will also be shrunk heavily.

This is not meant to be the final sophisticated opponent model. It is a stable first in season layer that can be improved later without touching the preseason pipeline.


In [7]:
opponent_features = team_summary[
    [
        "team",
        "offense_points_above_avg",
        "defense_points_saved_vs_avg"
    ]
].rename(
    columns={
        "team": "opponent",
        "offense_points_above_avg": "opp_offense_points_above_avg",
        "defense_points_saved_vs_avg": "opp_defense_points_saved_vs_avg"
    }
)

team_games_adj = team_games.merge(
    opponent_features,
    on="opponent",
    how="left"
)

team_games_adj["offense_above_avg_game"] = (
    team_games_adj["points_for"]
    - league_points_per_team_game
)

team_games_adj["defense_saved_vs_avg_game"] = (
    league_points_per_team_game
    - team_games_adj["points_against"]
)


OPPONENT_ADJUSTMENT_STRENGTH = 0.50

team_games_adj["opponent_adjusted_offense"] = (
    team_games_adj["offense_above_avg_game"]
    + (
        OPPONENT_ADJUSTMENT_STRENGTH
        * team_games_adj["opp_defense_points_saved_vs_avg"]
    )
)

team_games_adj["opponent_adjusted_defense"] = (
    team_games_adj["defense_saved_vs_avg_game"]
    + (
        OPPONENT_ADJUSTMENT_STRENGTH
        * team_games_adj["opp_offense_points_above_avg"]
    )
)

display(
    team_games_adj[
        [
            "team",
            "opponent",
            "points_for",
            "points_against",
            "offense_above_avg_game",
            "opponent_adjusted_offense",
            "defense_saved_vs_avg_game",
            "opponent_adjusted_defense"
        ]
    ].head(12)
)


,team,opponent,points_for,points_against,offense_above_avg_game,opponent_adjusted_offense,defense_saved_vs_avg_game,opponent_adjusted_defense
0,ARI,LAC,26.0,14.0,1.28125,0.640625,10.71875,5.359375
1,ATL,PIT,13.0,20.0,-11.71875,-5.859375,4.71875,2.359375
2,BAL,IND,41.0,23.0,16.28125,8.140625,1.71875,0.859375
3,BUF,HOU,36.0,31.0,11.28125,5.640625,-6.28125,-3.140625
4,CAR,CHI,37.0,59.0,12.28125,6.140625,-34.28125,-17.140625
5,CHI,CAR,59.0,37.0,34.28125,17.140625,-12.28125,-6.140625
6,CIN,TB,33.0,27.0,8.28125,4.140625,-2.28125,-1.140625
7,CLE,JAX,10.0,34.0,-14.71875,-7.359375,-9.28125,-4.640625
8,DAL,NYG,20.0,28.0,-4.71875,-2.359375,-3.28125,-1.640625
9,DEN,KC,10.0,31.0,-14.71875,-7.359375,-6.28125,-3.140625


# Aggregate Opponent Adjusted Features

The adjusted game level values are averaged to the team level.

Later in the season, this naturally becomes a multi game average.


In [8]:
adjusted_summary = (
    team_games_adj
    .groupby("team")
    .agg(
        opponent_adjusted_offense=(
            "opponent_adjusted_offense",
            "mean"
        ),
        opponent_adjusted_defense=(
            "opponent_adjusted_defense",
            "mean"
        )
    )
    .reset_index()
)

team_features = team_summary.merge(
    adjusted_summary,
    on="team",
    how="left"
)

team_features["opponent_adjusted_net"] = (
    team_features["opponent_adjusted_offense"]
    + team_features["opponent_adjusted_defense"]
)

display(
    team_features.sort_values(
        "opponent_adjusted_net",
        ascending=False
    )
)


,team,games_played,points_for,points_against,point_margin,points_for_per_game,points_against_per_game,point_diff_per_game,offense_points_above_avg,defense_points_saved_vs_avg,raw_net_points_above_avg,opponent_adjusted_offense,opponent_adjusted_defense,opponent_adjusted_net
14,JAX,1,34.0,10.0,24.0,34.0,10.0,24.0,9.28125,14.71875,24.0,4.640625,7.359375,12.0
5,CHI,1,59.0,37.0,22.0,59.0,37.0,22.0,34.28125,-12.28125,22.0,17.140625,-6.140625,11.0
15,KC,1,31.0,10.0,21.0,31.0,10.0,21.0,6.28125,14.71875,21.0,3.140625,7.359375,10.5
28,SF,1,27.0,7.0,20.0,27.0,7.0,20.0,2.28125,17.71875,20.0,1.140625,8.859375,10.0
2,BAL,1,41.0,23.0,18.0,41.0,23.0,18.0,16.28125,1.71875,18.0,8.140625,0.859375,9.0
20,MIN,1,39.0,22.0,17.0,39.0,22.0,17.0,14.28125,2.71875,17.0,7.140625,1.359375,8.5
18,LV,1,27.0,13.0,14.0,27.0,13.0,14.0,2.28125,11.71875,14.0,1.140625,5.859375,7.0
24,NYJ,1,23.0,10.0,13.0,23.0,10.0,13.0,-1.71875,14.71875,13.0,-0.859375,7.359375,6.5
0,ARI,1,26.0,14.0,12.0,26.0,14.0,12.0,1.28125,10.71875,12.0,0.640625,5.359375,6.0
23,NYG,1,28.0,20.0,8.0,28.0,20.0,8.0,3.28125,4.71875,8.0,1.640625,2.359375,4.0


# Early Season Shrinkage

The weekly model should not overreact to one or two games.

We therefore shrink the new in season evidence toward zero, where zero means "league average in season evidence."

The preseason model itself will remain the actual prior in the next notebook.

This shrinkage only controls how much confidence we place in the observed in season performance before blending it with preseason team strength.

The current rule uses:

`reliability = games_played / (games_played + PRIOR_GAME_EQUIVALENT)`

With a prior equivalent of 5 games:

- after 1 game: reliability = 1 / 6 = 16.7%
- after 2 games: reliability = 2 / 7 = 28.6%
- after 4 games: reliability = 4 / 9 = 44.4%
- after 8 games: reliability = 8 / 13 = 61.5%

This is intentionally conservative. The value can later be estimated historically rather than hard coded.


In [9]:
PRIOR_GAME_EQUIVALENT = 5.0

team_features["inseason_reliability"] = (
    team_features["games_played"]
    / (
        team_features["games_played"]
        + PRIOR_GAME_EQUIVALENT
    )
)

team_features["shrunk_offense_signal"] = (
    team_features["opponent_adjusted_offense"]
    * team_features["inseason_reliability"]
)

team_features["shrunk_defense_signal"] = (
    team_features["opponent_adjusted_defense"]
    * team_features["inseason_reliability"]
)

team_features["shrunk_net_signal"] = (
    team_features["shrunk_offense_signal"]
    + team_features["shrunk_defense_signal"]
)

display(
    team_features[
        [
            "team",
            "games_played",
            "inseason_reliability",
            "opponent_adjusted_offense",
            "shrunk_offense_signal",
            "opponent_adjusted_defense",
            "shrunk_defense_signal",
            "shrunk_net_signal"
        ]
    ].sort_values(
        "shrunk_net_signal",
        ascending=False
    )
)


,team,games_played,inseason_reliability,opponent_adjusted_offense,shrunk_offense_signal,opponent_adjusted_defense,shrunk_defense_signal,shrunk_net_signal
14,JAX,1,0.166667,4.640625,0.773438,7.359375,1.226562,2.000000
5,CHI,1,0.166667,17.140625,2.856771,-6.140625,-1.023438,1.833333
15,KC,1,0.166667,3.140625,0.523438,7.359375,1.226562,1.750000
28,SF,1,0.166667,1.140625,0.190104,8.859375,1.476562,1.666667
2,BAL,1,0.166667,8.140625,1.356771,0.859375,0.143229,1.500000
20,MIN,1,0.166667,7.140625,1.190104,1.359375,0.226562,1.416667
18,LV,1,0.166667,1.140625,0.190104,5.859375,0.976562,1.166667
24,NYJ,1,0.166667,-0.859375,-0.143229,7.359375,1.226562,1.083333
0,ARI,1,0.166667,0.640625,0.106771,5.359375,0.893229,1.000000
23,NYG,1,0.166667,1.640625,0.273438,2.359375,0.393229,0.666667


# Standardized In Season Signals

The preseason team strength model works in standardized strength space.

To make the weekly features easier to blend later, we also create z-scores across the 32 teams.

These are not yet the updated team strength ratings.

They are simply standardized measures of what the completed games have told us so far.


In [10]:
def safe_zscore(series):
    std = series.std(ddof=0)

    if pd.isna(std) or std == 0:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (
        series - series.mean()
    ) / std


team_features["inseason_offense_z"] = safe_zscore(
    team_features["shrunk_offense_signal"]
)

team_features["inseason_defense_z"] = safe_zscore(
    team_features["shrunk_defense_signal"]
)

team_features["inseason_net_z"] = safe_zscore(
    team_features["shrunk_net_signal"]
)

display(
    team_features[
        [
            "team",
            "shrunk_offense_signal",
            "inseason_offense_z",
            "shrunk_defense_signal",
            "inseason_defense_z",
            "shrunk_net_signal",
            "inseason_net_z"
        ]
    ].sort_values(
        "inseason_net_z",
        ascending=False
    )
)


,team,shrunk_offense_signal,inseason_offense_z,shrunk_defense_signal,inseason_defense_z,shrunk_net_signal,inseason_net_z
14,JAX,0.773438,0.827613,1.226562,1.312477,2.000000,1.694147
5,CHI,2.856771,3.056873,-1.023438,-1.095124,1.833333,1.552968
15,KC,0.523438,0.560102,1.226562,1.312477,1.750000,1.482379
28,SF,0.190104,0.203420,1.476562,1.579988,1.666667,1.411789
2,BAL,1.356771,1.451806,0.143229,0.153262,1.500000,1.270610
20,MIN,1.190104,1.273465,0.226562,0.242432,1.416667,1.200021
18,LV,0.190104,0.203420,0.976562,1.044966,1.166667,0.988252
24,NYJ,-0.143229,-0.153262,1.226562,1.312477,1.083333,0.917663
0,ARI,0.106771,0.114250,0.893229,0.955795,1.000000,0.847073
23,NYG,0.273438,0.292590,0.393229,0.420773,0.666667,0.564716


# Sanity Checks

These checks are especially important after Week 1.

We want:

- 32 teams represented once the prior week is complete
- no missing team level feature values
- offense and defense signals centered around zero
- no accidental inclusion of target week results


In [11]:
print("Teams in feature table:", len(team_features))
print("Unique teams:", team_features["team"].nunique())

feature_columns = [
    "opponent_adjusted_offense",
    "opponent_adjusted_defense",
    "shrunk_offense_signal",
    "shrunk_defense_signal",
    "shrunk_net_signal",
    "inseason_offense_z",
    "inseason_defense_z",
    "inseason_net_z"
]

print(
    "Missing feature values:",
    team_features[feature_columns].isna().sum().sum()
)

print()
print(
    "Mean shrunk offense signal:",
    round(
        team_features["shrunk_offense_signal"].mean(),
        6
    )
)

print(
    "Mean shrunk defense signal:",
    round(
        team_features["shrunk_defense_signal"].mean(),
        6
    )
)

if len(team_features) != 32:
    print()
    print(
        "WARNING: The feature table does not yet contain "
        "all 32 teams."
    )
    print(
        "This is expected if the previous week's games "
        "are not all final."
    )


Teams in feature table: 32
Unique teams: 32
Missing feature values: 0

Mean shrunk offense signal: -0.0
Mean shrunk defense signal: 0.0


# Save In Season Feature Table

This output will be consumed by:

`04_Weekly_Team_Strength.ipynb`

The file contains only the new in season evidence. It does not overwrite or replace the preseason team strength table.


In [12]:
output_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

team_features = team_features.sort_values(
    "team"
).reset_index(drop=True)

team_features.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)


Saved: ..\..\data\processed\weekly\week_02_inseason_features.parquet


# Interpretation

At this point we have two separate sources of information:

## Frozen preseason information

Created by the original `00–09` model pipeline and left untouched.

## New in-season evidence

Created here from games completed before the target week.

The next notebook, `04_Weekly_Team_Strength.ipynb`, will combine these two sources carefully.

The preseason rating remains the prior.

The in season features provide new evidence.

After Week 1, the in season signal is deliberately weak because the sample size is only one game per team. As more games are played, the reliability factor will naturally increase.
